In [29]:
suppressPackageStartupMessages(library(SingleCellExperiment))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(argparse))
suppressPackageStartupMessages(library(edgeR))
suppressPackageStartupMessages(library(ggrastr))
suppressPackageStartupMessages(library(scran))

######################
## Define arguments ##
######################

# p <- ArgumentParser(description='')
# p$add_argument('--sce',             type="character",                               help='SingleCellExperiment file')
# p$add_argument('--metadata',        type="character",                               help='Cell metadata file')
# p$add_argument('--stage',           type="character",                               help='Stages to include')
# p$add_argument('--variable',        type="character",                               help='Variable to perform differential testing over')
# p$add_argument('--tdTom_corr',      type="character",                               help='Keep tdTom+ cells in WT samples (otherwise removed)')
# p$add_argument('--method',          type="character",                               help='Method to use for DEGs, options: singlecell, pseudobulk, pseudobulkreplicates (pseudobulk with replicates from within each sample)')
# p$add_argument('--outdir',          type="character",                               help='Output file')

# args <- p$parse_args(commandArgs(TRUE))

#####################
## Define settings ##
#####################
here::i_am("processing/1_create_seurat_rna.R")
source(here::here("settings.R"))
source(here::here("utils.R"))
source(here::here("mapping/run/mnn/mapping_functions.R"))
test = TRUE

if(test){
## START TEST ##
    args = list()
args$sce <- io$rna.sce
args$metadata <- io$metadata
args$stage <- c('E8.5')#, 'E8.5', 'E9.5')
args$variable = 'pass_rnaQC'
args$metadata <- paste0(io$basedir,"/results/rna/mapping/sample_metadata_after_mapping.txt.gz")
args$tdTom_corr = 'False'
args$method = 'pseudobulk'
args$outdir <- paste0(io$basedir,"/results/rna/differential/test")
## END TEST ##
}

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/01_Eomes_RNA/code



In [30]:
# If passing multiple timepoints split in vector
args$stage = strsplit(args$stage, "_")[[1]] 

dir.create(args$outdir, recursive=TRUE, showWarnings = FALSE)

In [31]:
# Plotting function
gg_volcano_plot <- function(celltype_i='Allantois', top_genes=20, xlim=5, ylim=40, label_groups = NULL) {
  to.plot = de.results[variable == celltype_i]
  negative_hits <- to.plot[sig==TRUE & logFC<0,gene]
  positive_hits <- to.plot[sig==TRUE & logFC>0,gene]
  all <- nrow(to.plot[!is.na(sig)])
  to.plot = to.plot[,logFC_plot:=ifelse(logFC<=-xlim, -xlim, ifelse(logFC>=xlim, xlim, logFC))] %>%
               .[,log10_padj_fdr:=ifelse(-log10(padj_fdr)>=ylim, ylim, -log10(padj_fdr))]
  
  # if (is.null(xlim))
  #   xlim <- max(abs(to.plot$logFC), na.rm=T)
  # if (is.null(ylim))
  #   ylim <- max(-log10(to.plot$padj_fdr+1e-100), na.rm=T)
  
  to.plot <- to.plot[!is.na(logFC) & !is.na(padj_fdr)] %>% .[order(-abs(logFC))]
  label_genes = c(head(to.plot[sig==T & logFC<=0, gene],n=top_genes), head(to.plot[sig==T & logFC>=0, gene],n=top_genes))
  
  p <- ggplot(to.plot, aes(x=logFC_plot, y=log10_padj_fdr)) +
    labs(x="Log fold change", y=expression(paste("-log"[10],"(q.value)"))) +
    ggrastr::geom_point_rast(aes(color=sig, size=sig)) +
    # geom_hline(yintercept = -log10(opts$threshold_fdr), color="blue") +
    geom_segment(aes(x=0, xend=0, y=0, yend=ylim*1.05), color="orange", size=0.5) +
    scale_color_manual(values=c("black","red")) +
    scale_size_manual(values=c(0.5,1)) +
    scale_x_continuous(limits=c(-xlim,xlim)) +
    scale_y_continuous(limits=c(0,ylim*1.16)) +
    annotate("text", x=0, y=ylim*1.1, size=6, label=sprintf("(%d)", all)) +
    annotate("text", x=-xlim*0.9, y=ylim*1.15, size=6, label=sprintf("%d (-)",length(negative_hits))) +
    annotate("text", x=xlim*0.9, y=ylim*1.15, size=6, label=sprintf("%d (+)",length(positive_hits))) +
    ggrepel::geom_text_repel(data=to.plot[gene %in% label_genes],
                                    aes(x=logFC_plot, y=log10_padj_fdr, label=gene), max.overlaps=Inf, size=5) +
    theme_classic() +
    theme(
       axis.text = element_text(color='black'),
      # axis.title = element_text(size=rel(1.0), color='black'),
      text=element_text(size=15),
      legend.position="none"
    ) + 
    ggtitle(celltype_i)
  
  
  if (length(label_groups)>0) {
    p <- p +
      annotate("text", x=-4, y=0, size=4, label=sprintf("Up in %s",label_groups[2])) +
      annotate("text", x=4, y=0, size=4, label=sprintf("Up in %s",label_groups[1]))
  }
  
  return(p)
}

# Single cell DEG function

doDiffExpr <- function(sce, groups, min_detection_rate_per_group = 0.35) {
    
  # Sanity checks
  if (!is(sce, "SingleCellExperiment")) stop("'sce' has to be an instance of SingleCellExperiment")
  stopifnot(length(groups)==2)

  # Filter genes by detection rate per group
  cdr_A <- rowMeans(counts(sce[,sce$tdTom_corr==groups[1]])>0) >= min_detection_rate_per_group
  cdr_B <- rowMeans(counts(sce[,sce$tdTom_corr==groups[2]])>0) >= min_detection_rate_per_group
  out <- .edgeR(sce[cdr_B | cdr_A,]) %>% .[,log_padj_fdr:= -log10(padj_fdr)]
  
  return(out)
}


.edgeR <- function(sce) {
  
  # Convert SCE to DGEList
  
  sce_edger <- scran::convertTo(sce, type="edgeR")
 # sce_edger$samples$norm.factors = sce_edger$samples$sizeFactor
    
  # Define design matrix (with intercept)
  #cdr <- colMeans(logcounts(sce)>0)
  # if overlap in pools between KO and WT, use pool in model
  # Should I even run this if there's no overlap in pool? or just always use without pool?
  meta = as.data.table(colData(sce))
  if(length(intersect(unique(meta[tdTom_corr==TRUE,pool]), unique(meta[tdTom_corr==FALSE,pool])))>0 & length(unique(meta$pool))>1){
      design <- model.matrix(~sce$pool+sce$tdTom_corr)

      # Estimate dispersions
      sce_edger  <- estimateDisp(sce_edger,design)

      # Fit GLM
      fit <- glmQLFit(sce_edger,design)

      # Likelihood ratio test
      lrt <- glmQLFTest(fit)

      # Construct output data.frame
      out <- topTags(lrt, n=nrow(lrt))$table %>% as.data.table(keep.rownames=T) %>%
        setnames(c("gene","logFC","logCPM","LR","p.value","padj_fdr")) %>%
        .[,c("logCPM","LR"):=NULL]  
  }else{
      out = data.table("gene" = NA, "logFC" = NA, "p.value" = NA, "padj_fdr" = NA)
  }
  
  return(out)
}

In [32]:
##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadata) %>%
   .[pass_rnaQC==TRUE & doublet_call==FALSE & stage %in% args$stage] %>%
   .[,pool:=stringr::str_replace_all(sample,opts$sample2pool)] %>% 
   .[,variable := .[[args$variable]]]

# discard Tomato+ cells from Tomato- samples
if(args$tdTom_corr!='True'){
    sample_metadata = sample_metadata[tdTom==tdTom_corr]
}

In [33]:
# if(test){
#     sample_metadata = sample_metadata[sample(1:nrow(sample_metadata), nrow(sample_metadata)/4)]
# }

###############
## Load data ##
###############

# Load RNA expression data as SingleCellExperiment object
sce <- load_SingleCellExperiment(args$sce, cells=sample_metadata$cell, normalise = TRUE)

# Add sample metadata as colData
colData(sce) <- sample_metadata %>% tibble::column_to_rownames("cell") %>% DataFrame

In [34]:
head(sample_metadata$pass_rnaQC

ERROR: Error in parse(text = x, srcfile = src): <text>:2:0: unexpected end of input
1: head(sample_metadata$pass_rnaQC
   ^


In [ ]:
###################
## exclude genes ##
###################

# haemoglobin genes show up as differential expressed in all cell-types when there is a differential abundance in erythroids
# We hypothesise this is due to bursting of the erythroids, resulting in large amount of ambient haemoglobin RNA.
gene_biomart = fread('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/Mmusculus_genes_BioMart.87.txt')
all_genes = gene_biomart$symbol
haem_genes = all_genes[c(grep('Hba', all_genes),
                         grep('Hbb', all_genes))]

# Genes on the y-chr can be differential due to the injected ESC being male, while the host can be either male or female
y_genes = gene_biomart[chr=='chrY', symbol]

# Some genes on the x-chr can be differential for the same reason
# Xist and Tsix are known to often be differentially expressed 
x_genes = c('Xist', 'Tsix')

# Additionally there are some paternally or maternally expressed genes to remove
imprint = gene_biomart[c(grep('maternally', gene_biomart$description),
                       grep('paternally', gene_biomart$description)), symbol]
imprint = c(imprint, 'Grb10', 'Nnat')

# Remove all other genes that are not informative, e.g. mitochondrial/ribosomal/predicted genes etc
non_informative = all_genes[grep("*Rik|^Gm|^mt-|^Rps|^Rpl|^Olfr", all_genes)]

# Genes differential from WT injection experiment
#WT_inj = fread('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/WT_injection/old/tomato_DEGs_edgeR.csv')

# Tomato-td is differentially expressed, because it is only present in the KO cells
excl_genes = intersect(c(haem_genes, y_genes, x_genes, imprint, non_informative, 'tomato-td'), rownames(sce))


sce = sce[!rownames(sce) %in% excl_genes,]

In [ ]:
# Pseudobulk 

if(args$method=='pseudobulk'){
    # Using 'celltype' and 'sample' as our two factors; each column of the output
    # corresponds to one unique combination of these two factors.
    summed <- aggregateAcrossCells(sce, 
        id=colData(sce)[,c("variable", "sample", "tdTom_corr")])

    summed.filt <- summed[,summed$ncells >= 10]

    de.results <- pseudoBulkDGE(summed.filt, 
        label=summed.filt$variable,
        design=~factor(pool) + tdTom_corr,
        coef="tdTom_corrTRUE",
        condition=summed.filt$tdTom_corr 
        ) %>% unlist() %>% 
            as.data.table(keep.rownames=T) %>%
            .[,`:=`(gene=str_split(rn, '\\.') %>% map_chr(2), # extract gene names
                    variable=str_split(rn, '\\.') %>% map_chr(1), # extract cell type
                    rn=NULL, 
                    stage=paste(args$stage, collapse='_'), # add stage
                    sig=ifelse(abs(logFC)>=1 & FDR<=0.05, TRUE, FALSE))] %>%  # significance
            .[!is.na(logFC),] %>%
            setnames('FDR', 'padj_fdr')
    
    # Make plots
    plots = lapply(unique(de.results$variable),gg_volcano_plot, ylim=20, xlim=5)
}

In [27]:
head(de.results)

logFC,logCPM,F,PValue,padj_fdr,gene,variable,stage,sig
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<lgl>
0.064438216,-0.39921346,0.04581985,0.835640737,0.88226525,5,E8,E8.5,FALSE
-0.063729342,3.34351619,0.08962193,0.774253544,0.83371094,5,E8,E8.5,FALSE
-0.049346565,8.18338911,0.81017182,0.393297701,0.49466042,5,E8,E8.5,FALSE
0.246370607,5.10880186,16.80089621,0.003145784,0.01110311,5,E8,E8.5,FALSE
-0.008328975,7.59543538,0.02107042,0.888031652,0.92026249,5,E8,E8.5,FALSE
0.706435680,0.04654821,6.41698532,0.033947126,0.07009807,5,E8,E8.5,FALSE


In [23]:
plots